<a href="https://colab.research.google.com/github/cloudbloqavi/hybrid-search-rag/blob/main/RAG_%2B_Hybrid_Search_with_Crew_AI%2C_NeonDb%2C_Qdrant_and_Gemini_a_real_world_scenario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install psycopg2-binary qdrant-client

In [ ]:
!pip install crewai

In [ ]:
!pip install crewai-tools

In [ ]:
!pip install langchain-google-genai

In [ ]:
import os
import psycopg2
from qdrant_client import QdrantClient, models
from qdrant_client.models import Prefetch, Query, Fusion
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

GEMINI_API_KEY = "" # add  you gemini API key

# --- 1. Neon Database and Qdrant Setup ---
# PostgreSQL connection details
DB_HOST = os.environ.get("DB_HOST", "postgres host")
DB_NAME = os.environ.get("DB_NAME", "database name")
DB_USER = os.environ.get("DB_USER", "db user name")
DB_PASSWORD = os.environ.get("DB_PASSWORD", "db user password")

# Qdrant connection (get it from qdrant cloud, free cluster)
QDRANT_URL = ""
QDRANT_API_KEY = ""
COLLECTION_NAME = "food_recipes"

# Sentence Transformer model for embeddings
EMBEDDING_MODEL_NAME = 'models/gemini-embedding-exp-03-07'

# Initialize clients
qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)
# This is the new embedding model object
embedding_model = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL_NAME, google_api_key=GEMINI_API_KEY)
# TF-IDF vectorizer remains the same
tfidf_vectorizer = TfidfVectorizer()


# --- 2. Data Indexing ---

def setup_database_and_qdrant():
    """
    Sets up the PostgreSQL database with sample data and creates the Qdrant collection with hybrid indexing.
    """
    # Connect to PostgreSQL
    conn = psycopg2.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)
    cur = conn.cursor()

    # Create table and insert some sample data
    cur.execute("""
        CREATE TABLE IF NOT EXISTS recipes (
            id SERIAL PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            description TEXT,
            cuisine VARCHAR(100),
            season VARCHAR(50)
        );
    """)

    # Clear existing data to avoid duplicates on re-runs
    cur.execute("TRUNCATE TABLE recipes RESTART IDENTITY;")

    sample_recipes = [
        ('Spicy Thai Green Curry', 'A classic Thai green curry with chicken, coconut milk, and fresh basil.', 'Thai', 'All'),
        ('Hearty Winter Stew', 'A rich and comforting beef stew with root vegetables, perfect for a cold day.', 'American', 'Winter'),
        ('Summer Berry Salad', 'A light and refreshing salad with mixed greens, fresh berries, and a vinaigrette dressing.', 'Fusion', 'Summer'),
        ('Pad Thai', 'A popular Thai stir-fried noodle dish with shrimp, tofu, and peanuts.', 'Thai', 'All'),
        ('Pumpkin Spice Soup', 'A creamy and flavorful soup made with roasted pumpkin and warm spices.', 'American', 'Autumn')
    ]

    for recipe in sample_recipes:
        cur.execute("INSERT INTO recipes (name, description, cuisine, season) VALUES (%s, %s, %s, %s)", recipe)

    conn.commit()
    print("Sample data inserted into the 'recipes' table.")

    # Setup Qdrant collection for hybrid search
    qdrant_client.recreate_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "semantic-vector": models.VectorParams(size=3072, distance=models.Distance.COSINE)
        },
        sparse_vectors_config={
            "keyword-vector": models.SparseVectorParams(
                index=models.SparseIndexParams(
                    on_disk=False
                )
            )
        }
    )
    print("Qdrant collection recreated.. Indexing started")

    # Index data into Qdrant
    cur.execute("SELECT id, name, description, cuisine, season FROM recipes;")
    recipes = cur.fetchall()

    # Fit the TF-IDF vectorizer on all recipe descriptions
    all_descriptions = [recipe[2] for recipe in recipes]
    tfidf_vectorizer.fit(all_descriptions)

    points = []
    for recipe_id, name, description, cuisine, season in recipes:
        # Dense vector for semantic meaning
        dense_embedding = embedding_model.embed_query(description)

        # Sparse vector for keywords
        sparse_vector = tfidf_vectorizer.transform([description])
        sparse_indices = sparse_vector.indices.tolist()
        sparse_values = sparse_vector.data.tolist()

        points.append(
            models.PointStruct(
                id=recipe_id,
                vector={
                    "semantic-vector": dense_embedding,
                    "keyword-vector": models.SparseVector(
                        indices=sparse_indices,
                        values=sparse_values
                    )
                },
                payload={"name": name, "description": description, "cuisine": cuisine, "season": season}
            )
        )

    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
        wait=True
    )
    print(f"Indexed {len(recipes)} recipes into Qdrant collection '{COLLECTION_NAME}'.")

    cur.close()
    conn.close()

In [ ]:
!pip install -U qdrant-client

In [ ]:
# --- 3. Crew AI Agent and Task ---
from crewai import LLM

llm = LLM(
    model="gemini/gemini-2.0-flash", # Or another suitable Gemini model
    temperature=0.7,
    api_key=GEMINI_API_KEY
)

class RecipeSearchTool(BaseTool):
    name: str = "Recipe Search Tool"
    description: str = "Searches for recipes in the Qdrant database using a query. Use this to find recipes based on user preferences."

    def _run(self, query: str) -> str:
        """
        Performs a hybrid search in Qdrant.
        """
        # Dense vector for the query
        dense_query_embedding = embedding_model.embed_query(query)

        # Sparse vector for the query
        sparse_query_vector = tfidf_vectorizer.transform([query])
        sparse_query_indices = sparse_query_vector.indices.tolist()
        sparse_query_values = sparse_query_vector.data.tolist()

        # 1. Define the prefetch operations for both dense and sparse searches
        prefetch_queries = [
            models.Prefetch(
                query=dense_query_embedding,
                using="semantic-vector",
                limit=5,
            ),
            models.Prefetch(
                query=models.SparseVector(
                    indices=sparse_query_indices,
                    values=sparse_query_values
                ),
                using="keyword-vector",
                limit=5,
            )
        ]

        # 2. Use `models.FusionQuery` to explicitly request fusion on the prefetched results
        search_results = qdrant_client.query_points(
            collection_name=COLLECTION_NAME,
            prefetch=prefetch_queries,
            query=models.FusionQuery(
                fusion=models.Fusion.RRF,
            ),
            limit=3,
            with_payload=True,
        )

        # The results are in the `points` attribute of the response object
        if not search_results.points:
            return "Found no matching recipes."

        results_str = "Found recipes:\n"
        for result in search_results.points:
            results_str += f"- {result.payload['name']}: {result.payload['description']} (Cuisine: {result.payload['cuisine']}, Season: {result.payload['season']})\n"

        return results_str

# Define the agent
recipe_expert = Agent(
    role='Recipe Expert',
    goal='Find the best recipes for the user based on their preferences.',
    backstory='You are an expert in all types of cuisine and know the best recipes for any occasion.',
    verbose=True,
    llm=llm,
    allow_delegation=False,
    tools=[RecipeSearchTool()]
)

# Define the task
recipe_task = Task(
    description='Find a recipe based on the query: \"{query}\"',
    expected_output='A friendly response with the recommended recipe(s) and why they are a good fit.',
    agent=recipe_expert
)

# --- 4. Main Execution Block ---

if __name__ == "__main__":
    print("--- Setting up Database and Qdrant ---")
    setup_database_and_qdrant()
    print("\n--- Setup Complete ---")

    # Create and run the crew
    recipe_crew = Crew(
        agents=[recipe_expert],
        tasks=[recipe_task],
        process=Process.sequential
    )

    # Get user input
    user_query = input("\nWhat kind of recipe are you looking for? (e.g., 'best thai recipe for dinner during winter'): ")

    result = recipe_crew.kickoff(inputs={'query': user_query})

    print("\n--- Here's the recommendation ---")
    print(result)